# Parte 1. Introducción a la visualización de datos de GLOBE

Esta lección muestra cómo investigar datos de mosquitos y cobertura terrestre de GLOBE, calcular estadísticas y crear gráficos y mapas a escala mundial, estatal y de condado.

## Introducción a las herramientas

* **Python** es un lenguaje de programación popular que se utiliza para analizar datos, automatizar tareas, crear software, entrenar modelos de aprendizaje automático y mucho más. Lo utilizaremos para analizar datos de GLOBE Mosquito Habitat Mapper y GLOBE Land Cover.
* **Google Colab** es un entorno gratuito de programación en la nube que permite ejecutar Python desde el navegador. *¡No es necesario instalar ni descargar nada!*

**Para ejecutar el código:**

Cada bloque de código se denomina celda. Para ejecutar una celda, pasa el cursor sobre ella y haz clic en la flecha de la esquina superior izquierda, o haz clic dentro de la celda y presiona `Shift + Enter`.

Nota: La primera vez que ejecutes un bloque de código, Google Colab mostrará el mensaje `Warning: This notebook was not authored by Google`. Haz clic en `Run Anyway`.

In [ ]:
# Importar las bibliotecas
import pandas as pd                           # Trabajar con datos
pd.set_option("display.max_columns", None)    # Mostrar todas las columnas
import geopandas as gpd                       # Trabajar con datos espaciales
import numpy as np                            # Trabajar con números
from datetime import date                     # Trabajar con fechas
import matplotlib.pyplot as plt               # Crear gráficos
from matplotlib.colors import to_rgb          # Obtener colores
import branca.colormap as cm                  # Crear escalas de colores
import seaborn as sns                         # Usar paletas y crear gráficos
import folium                                 # Crear mapas interactivos
from PIL import Image                         # Obtener y mostrar imágenes
import requests                               # Obtener información de enlaces
from io import BytesIO                        # Trabajar con entradas y salidas

En el código anterior importamos bibliotecas de Python, que amplían las posibilidades de nuestro código. Cada biblioteca incluye muchas funciones diseñadas para realizar tareas específicas, como cargar un conjunto de datos, hacer cálculos, crear un gráfico y mucho más.

In [ ]:
# Este es un comentario, identificado por el símbolo # al inicio
# Los comentarios explican el código, pero no afectan su ejecución

# Mostrar la fecha de hoy
today = date.today()
print(f"La fecha de hoy es {today}.")

Una gran ventaja de Google Colab es que puedes escribir código en Python y ver el resultado directamente en el navegador.

## Mosquitos alrededor del mundo con GLOBE

Cargaremos los datos directamente desde el enlace. Todo permanecerá dentro de Google Colab en el navegador y no se descargará nada en tu computadora. Esta es una versión procesada de los datos, con algunas columnas renombradas y ciertos errores corregidos.

Los datos provienen de GLOBE Observer y se obtuvieron mediante [la API](https://www.globe.gov/globe-data/globe-api). Puedes consultar todos los pasos utilizados para procesar y limpiar los datos en el [capítulo 1 del currículo de EMERGE](https://geo-di-lab.github.io/emerge-lessons/docs/ch1/lesson6.html).

In [ ]:
mosquito = gpd.read_file('https://github.com/geo-di-lab/emerge-lessons/raw/refs/heads/main/docs/data/globe_mosquito.zip')
mosquito.head()

Consulta la lista de columnas:

In [ ]:
mosquito.info()

¿Cuántas filas contiene el conjunto de datos?

In [ ]:
len(mosquito)

Hubo 43,012 contribuciones de ciencia participativa entre 2018 y 2024. Ahora veremos en cuántos países se enviaron datos.

In [ ]:
len(mosquito['CountryCode'].unique())

Veamos los tipos de hábitats, es decir, fuentes de agua, que registraron las personas participantes.

In [ ]:
# Tipos generales de fuentes de agua
mosquito["WaterSourceType"].value_counts()

Estos son los tipos generales de fuentes de agua que las personas participantes reportaron a NASA GLOBE. La mayoría de los datos recopilados parecen corresponder a recipientes artificiales. Veamos algunos de los tipos más específicos incluidos en la otra columna:

In [ ]:
# Tipos más específicos de fuentes de agua
mosquito["WaterSource"].value_counts()

Crearemos un gráfico circular con los tipos más generales de la columna `WaterSourceType`.

In [ ]:
# Algunas opciones de paletas de colores
display(sns.color_palette(palette="rainbow"))
display(sns.color_palette(palette="CMRmap"))
display(sns.color_palette(palette="BrBG"))
display(sns.color_palette(palette="cubehelix"))
display(sns.color_palette(palette="Set2"))
display(sns.color_palette(palette="Set3"))
display(sns.color_palette(palette="tab20"))

In [ ]:
# Gráfico circular de los tipos de fuentes de agua
types = (
    mosquito[["SiteId", "WaterSourceType"]]
    .groupby("WaterSourceType", as_index=False)
    .count()
)

# Crear el gráfico circular
plt.figure(figsize=(5, 5))
patches, texts = plt.pie(
    colors=sns.color_palette("Set2"),  # Nombre de la paleta elegida
    x=types["SiteId"]
)
plt.title(
    "Observaciones de mosquitos de GLOBE: "
    "tipos generales de fuentes de agua"
)
plt.legend(
    patches,
    types["WaterSourceType"],
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False
)
plt.show()

¿Cómo son estas fuentes de agua? A continuación mostraremos 10 fotografías:

In [ ]:
# Obtener 10 URL de fotografías del conjunto de datos
rows_with_photos = (
    mosquito.dropna(subset=["WaterSourcePhotoUrls"])
    .loc[mosquito["WaterSourcePhotoUrls"] != "rejected"]
    .reset_index()
    .sample(n=10, random_state=1)
)
url_list = (
    rows_with_photos["WaterSourcePhotoUrls"]
    .str.split("; ")
    .str[0]
    .tolist()
)
source_list = rows_with_photos["WaterSource"].tolist()

# Mostrar todas las fotografías
plt.figure(figsize=(20, 6))

for i, (url, title) in enumerate(zip(url_list, source_list)):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content))

    # Crear un gráfico con 2 filas y 5 columnas
    plt.subplot(2, 5, i + 1)
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

¿Cuál es el promedio de larvas registradas por país?

In [ ]:
mosquito_avg = mosquito.groupby('CountryCode')['LarvaeCountProcessed'].mean()
mosquito_avg

Crearemos un mapa que muestre el promedio de larvas por país. Los [límites generalizados de los países](https://hub.arcgis.com/datasets/esri::world-countries-generalized/about) provienen de Esri, Garmin y la Agencia Central de Inteligencia de Estados Unidos, mediante *The World Factbook*. Los límites están generalizados para que el procesamiento de los datos y la carga de las visualizaciones sean más rápidos.

Los códigos ISO alfa-3 provienen de la capa [World Countries](https://hub.arcgis.com/datasets/esri::world-countries/about), elaborada con datos de Esri, Garmin, la Agencia Central de Inteligencia de Estados Unidos y la Organización Internacional de Normalización (ISO).

In [ ]:
# Cargar los límites de los países
countries = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/"
    "raw/refs/heads/main/docs/data/"
    "world_countries_general.geojson"
).to_crs(epsg=4326)

# Agregar los datos de mosquitos correspondientes a cada país
mosquito_avg = countries.merge(
    mosquito_avg,
    left_on="iso3",
    right_on="CountryCode",
    how="left"
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Crear un mapa del promedio de larvas por país
mosquito_avg.plot(
    column="LarvaeCountProcessed",
    cmap="viridis",
    legend=True,
    vmin=0,
    vmax=50,
    ax=ax,
    missing_kwds={"color": "lightgrey"}
)
plt.title(
    "Observaciones de mosquitos de GLOBE: "
    "promedio de larvas"
)
ax.axis("off")
plt.show()

Ahora crearemos un mapa *interactivo* que muestre el total de observaciones de GLOBE por país.

In [ ]:
# Obtener el total de observaciones de GLOBE por país
mosquito_obs = (
    mosquito.groupby("CountryCode")
    .size()
    .reset_index(name="GLOBE_Observations")
)
mosquito_obs = countries.merge(
    mosquito_obs,
    left_on="iso3",
    right_on="CountryCode",
    how="left"
).dropna(subset=["GLOBE_Observations"])

In [ ]:
# Cargar una escala de colores que comienza en 1 y termina en 100
# El azul representa 100 observaciones o más; el verde y el amarillo,
# valores más cercanos a una observación
colors = cm.linear.YlGnBu_03.scale(1, 100)
colors

In [ ]:
map = folium.Map(
    location=[0, 0],
    zoom_start=3,
    tiles="CartoDB positron"
)

# Crear un mapa interactivo de las observaciones de GLOBE por país
folium.GeoJson(
    geo_data=mosquito_obs.to_json(),
    data=mosquito_obs,
    key_on="feature.properties.name",
    tooltip=folium.features.GeoJsonTooltip(
        fields=["name", "GLOBE_Observations"],
        aliases=["País:", "Observaciones:"]
    ),
    style_function=lambda feature: {
        "fillColor": colors(
            feature["properties"]["GLOBE_Observations"]
        ),
        "fillOpacity": 0.8,
        "color": "grey",
        "weight": 1
    }
).add_to(map)

display(map)

## Cobertura terrestre alrededor del mundo con GLOBE

Al igual que con los datos de mosquitos, cargaremos directamente desde el enlace los datos de cobertura terrestre de GLOBE.

In [ ]:
land_cover = gpd.read_file('https://github.com/geo-di-lab/emerge-lessons/raw/refs/heads/main/docs/data/globe_land_cover.zip')
land_cover.head()

In [ ]:
len(land_cover)

Una parte útil del conjunto de datos de cobertura terrestre son las clasificaciones MUC. MUC, o Clasificación Modificada de la UNESCO, es un sistema que incluye distintos tipos de uso del suelo y nos ayuda a comprender los hábitats de todo el mundo.

¿Cuáles son los códigos MUC más comunes en cada país?

In [ ]:
# Encontrar el MUC más común de cada país
muc = (
    land_cover.groupby("CountryCode")["MucDescription"]
    .apply(
        lambda x: (
            x.value_counts().idxmax()
            if not x.value_counts().empty
            else None
        )
    )
    .reset_index(name="MucDescription")
)

# Agregar la cantidad de observaciones con ese código MUC
muc["Count"] = (
    land_cover.groupby("CountryCode")["MucDescription"]
    .apply(lambda x: x.value_counts().max())
    .values
)

# Agregar el total de observaciones de GLOBE
muc["GLOBE_Observations"] = (
    land_cover.groupby("CountryCode").size().values
)

muc

In [ ]:
# Agregar los datos a los límites de los países
muc = countries.merge(
    muc,
    left_on="iso3",
    right_on="CountryCode",
    how="left"
)

# Crear categorías generales
# Estos valores permanecen en inglés porque deben coincidir con
# el contenido original de la columna MucDescription
muc_list = [
    "Barren",
    "Closed Forest",
    "Cultivated",
    "Herbaceous",
    "Open Water",
    "Trees",
    "Urban",
    "Wetlands",
    "Woodland"
]

# Simplificar las descripciones mediante las categorías anteriores
for muc_code in muc_list:
    muc.loc[
        muc["MucDescription"].str.contains(muc_code, na=False),
        "MucDescriptionShort"
    ] = muc_code

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

# Crear un mapa de los códigos MUC más comunes por país
muc.plot(
    column="MucDescriptionShort",
    cmap="viridis",
    legend=True,
    ax=ax,
    missing_kwds={
        "color": "lightgrey",
        "label": "Sin datos"
    },
    legend_kwds={
        "loc": "lower left",
        "frameon": False
    }
)
plt.title(
    "Cobertura terrestre de GLOBE: "
    "códigos MUC más comunes"
)
plt.show()

Observa las zonas grises. No se registraron observaciones de GLOBE en esos países, por lo que los eliminaremos del conjunto de datos para facilitar la visualización.

In [ ]:
muc = muc.dropna(subset=['GLOBE_Observations'])

Una característica importante del conjunto de datos de cobertura terrestre es que las personas participantes envían fotografías del área. Veamos algunas de estas imágenes.

In [ ]:
# Obtener la primera observación que incluya todas las fotografías
entry = land_cover.dropna(
    subset=[
        "DownwardPhotoUrl",
        "EastPhotoUrl",
        "NorthPhotoUrl",
        "SouthPhotoUrl",
        "WestPhotoUrl",
        "UpwardPhotoUrl",
        "Feature1PhotoUrl",
        "Feature2PhotoUrl",
        "Feature3PhotoUrl",
        "Feature4PhotoUrl"
    ]
).head(1)

url_list = []
col_list = []

for col in entry.columns:
    if "Url" in col:
        print(f"{col}: {entry[col].values[0]}")
        url_list.append(entry[col].values[0])
        col_list.append(col)

display(entry)

In [ ]:
# Mostrar todas las imágenes
plt.figure(figsize=(20, 6))

for i, (url, title) in enumerate(zip(url_list, col_list)):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content))

    # Crear un gráfico con 2 filas y 5 columnas
    plt.subplot(2, 5, i + 1)
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

### Desafío: crear un mapa interactivo de la cobertura terrestre

Al igual que en el mapa de observaciones de mosquitos, crearemos un mapa con la cantidad de observaciones de cobertura terrestre de GLOBE. Además del nombre del país y la cantidad de observaciones de GLOBE, agrega el código MUC más común, `MucDescriptionShort`, al cuadro de información que aparece al pasar el cursor sobre cada país.

Utiliza las siguientes columnas:

- `name` para el nombre del país
- `MucDescriptionShort` para el código MUC más común
- `GLOBE_Observations` para la cantidad de observaciones de GLOBE registradas y para determinar el color de cada país

En resumen, el color de cada país debe basarse en `GLOBE_Observations`. El cuadro emergente debe incluir el nombre del país, la cantidad de observaciones de GLOBE y el código MUC más común.

Puedes consultar como referencia el código del mapa interactivo de observaciones de mosquitos, que incluimos a continuación. La estructura técnica permanece en inglés porque utiliza los nombres de las columnas del conjunto de datos. Si necesitas ayuda, abre la sección de respuesta para ver una manera de crear el mapa.

```python
map = folium.Map(location=[0, 0], zoom_start=3, tiles="CartoDB positron")

# Crear un mapa interactivo de las observaciones de GLOBE por país
folium.GeoJson(
    geo_data=mosquito_obs.to_json(),
    data=mosquito_obs,
    key_on="feature.properties.name",
    tooltip=folium.features.GeoJsonTooltip(
        fields=["name", "GLOBE_Observations"],
        aliases=["País:", "Observaciones:"]
    ),
    style_function=lambda feature: {
        "fillColor": colors(
            feature["properties"]["GLOBE_Observations"]
        ),
        "fillOpacity": 0.8,
        "color": "grey",
        "weight": 1
    }
).add_to(map)

display(map)
```

In [ ]:
map = folium.Map(
    location=[0, 0],
    zoom_start=3,
    tiles="CartoDB positron"
)

# Completa estas cuatro variables
geo_data = None
map_data = None
tooltip_fields = []
tooltip_aliases = []

if (
    geo_data is None
    or map_data is None
    or not tooltip_fields
    or not tooltip_aliases
):
    print(
        "Completa geo_data, map_data, tooltip_fields "
        "y tooltip_aliases antes de crear el mapa."
    )
else:
    folium.GeoJson(
        geo_data=geo_data,
        data=map_data,
        key_on="feature.properties.name",
        tooltip=folium.features.GeoJsonTooltip(
            fields=tooltip_fields,
            aliases=tooltip_aliases
        ),
        style_function=lambda feature: {
            "fillColor": colors(
                feature["properties"]["GLOBE_Observations"]
            ),
            "fillOpacity": 0.8,
            "color": "grey",
            "weight": 1
        }
    ).add_to(map)

    display(map)

### Respuesta

In [ ]:
map = folium.Map(
    location=[0, 0],
    zoom_start=3,
    tiles="CartoDB positron"
)

folium.GeoJson(
    geo_data=muc.to_json(),
    data=muc,
    key_on="feature.properties.name",
    tooltip=folium.features.GeoJsonTooltip(
        fields=[
            "name",
            "GLOBE_Observations",
            "MucDescriptionShort"
        ],
        aliases=[
            "País:",
            "Observaciones:",
            "MUC más común:"
        ]
    ),
    style_function=lambda feature: {
        "fillColor": colors(
            feature["properties"]["GLOBE_Observations"]
        ),
        "fillOpacity": 0.8,
        "color": "grey",
        "weight": 1
    }
).add_to(map)

display(map)

## Mosquitos en tu estado

Ahora veremos los datos de GLOBE correspondientes a tu estado y, más adelante, a tu condado. Escribe a continuación el nombre de tu estado **en inglés**, tal como aparece en los datos del Censo de Estados Unidos.

Los archivos de límites provienen de la [Oficina del Censo de Estados Unidos](https://www.census.gov/geographies/mapping-files/time-series/geo/cartographic-boundary.html). Utilizaremos los mismos datos de mosquitos de la primera parte del cuaderno. Estos datos provienen de GLOBE Observer, se obtuvieron mediante [la API](https://www.globe.gov/globe-data/globe-api) y se procesaron en el [capítulo 1 de nuestro libro digital](https://geo-di-lab.github.io/emerge-lessons/docs/ch1/lesson6.html).

In [ ]:
# Escribe el nombre del estado en inglés
state_name = "Your State Name"

# Por ejemplo:
# state_name = "Florida" 

In [ ]:
# Cargar los límites de los condados
counties = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/"
    "raw/refs/heads/main/docs/data/us_counties.zip"
).to_crs("EPSG:4326")

# Filtrar todos los condados del estado seleccionado
state_counties = counties.loc[
    counties["STATE_NAME"] == state_name
]

# Obtener el límite estatal mediante la unión de los condados
state = state_counties[["geometry"]].dissolve()

state.plot()

In [ ]:
# Crear un mapa vacío centrado en el estado
map = folium.Map(tiles="Cartodb dark_matter")

# Obtener todos los datos de mosquitos de GLOBE dentro del estado
mosquito_state = (
    gpd.sjoin(
        mosquito,
        state,
        how="inner",
        predicate="intersects"
    )
    .drop(columns=["index_right"])
    .reset_index(drop=True)
)

# Agregar los límites de los condados al mapa
folium.GeoJson(
    state_counties.to_json(),
    name="Límites de los condados",
    tooltip=folium.features.GeoJsonTooltip(
        fields=["NAMELSAD"],
        aliases=["Condado:"]
    ),
    style_function=lambda feature: {
        "fillColor": "transparent",
        "color": "grey",
        "weight": 1
    }
).add_to(map)

# Agregar cada observación como un círculo verde
for idx, row in mosquito_state.iterrows():
    popup_content = (
        f"<b>Fecha:</b> {row['MeasuredDate']}<br>"
        f"<b>Fuente de agua:</b> {row['WaterSourceType']}<br>"
        f"<b>Longitud:</b> {row['MeasurementLongitude']}<br>"
        f"<b>Latitud:</b> {row['MeasurementLatitude']}"
    )

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        popup=folium.Popup(
            popup_content,
            max_width=300
        ),
        radius=6,
        color="black",
        weight=1,
        fillColor="lightgreen",
        fillOpacity=0.5
    ).add_to(map)

# Agregar una opción para ocultar los límites
folium.LayerControl().add_to(map)

# Acercar el mapa al estado
minx, miny, maxx, maxy = state.bounds.values[0]
bounds = [[miny, minx], [maxy, maxx]]
map.fit_bounds(bounds)

# Mostrar el mapa
display(map)

Comprueba si hay puntos dentro de tu condado. Si no los hay, busca un condado cercano que tenga puntos. Escribe a continuación el nombre del condado elegido **en inglés**:

In [ ]:
# Escribe el nombre del condado en inglés
county_name = "Your County Name"

# El nombre debe terminar con la palabra "County"

# Por ejemplo:
# county_name = "Broward County" 

## Mosquitos en tu condado

Obtén el contorno específico de tu condado:

In [ ]:
county = state_counties.loc[
    state_counties["NAMELSAD"] == county_name
]
county.plot()

Ahora utilizaremos esos límites para filtrar todos los puntos de GLOBE ubicados dentro de tu condado entre 2018 y 2024.

In [ ]:
data_county = mosquito.sjoin(
    county,
    how="inner",
    predicate="within"
)

num_total = len(data_county)

print(
    f"Se registraron {num_total} puntos de GLOBE dentro de "
    f"{county_name} entre 2018 y 2024."
)

num_eliminated = len(
    data_county[
        data_county["BreedingGroundEliminated"] == "true"
    ]
)

if num_total > 0:
    percentage = round(num_eliminated * 100 / num_total)
    print(
        f"De esos puntos, las personas participantes mitigaron "
        f"correctamente {num_eliminated} ({percentage} %). "
        "Esto reduce el riesgo de que los mosquitos ocupen ese "
        "lugar en el futuro."
    )
else:
    print(
        "No se encontraron puntos de GLOBE en el condado "
        "seleccionado. Elige un condado cercano con observaciones."
    )

Si no hay puntos de GLOBE en tu condado, elige un condado cercano que tenga al menos uno. Puedes identificarlo en el mapa estatal anterior.

In [ ]:
data_county.head()

¿Cuántas observaciones de mosquitos se enviaron cada año a GLOBE Observer desde tu condado? Crearemos un gráfico de barras para averiguarlo.

In [ ]:
# Agregar una columna con el año
data_county["MeasuredYear"] = (
    data_county["MeasuredAt"].dt.year
)

# Crear un gráfico de barras de las observaciones por año
years = (
    data_county[["SiteId", "MeasuredYear"]]
    .groupby("MeasuredYear", as_index=False)
    .count()
)
plt.bar(years["MeasuredYear"], years["SiteId"])
plt.title(
    "Observaciones de mosquitos por año",
    loc="left"
)
plt.title(county_name, loc="right")
plt.show()

Crearemos gráficos circulares de los tipos de fuentes de agua, tanto generales como específicos, donde se reportaron mosquitos dentro de este condado.

In [ ]:
# Contar cada tipo general de fuente de agua
types = (
    data_county[["SiteId", "WaterSourceType"]]
    .groupby("WaterSourceType", as_index=False)
    .count()
)

# Crear el gráfico circular
plt.figure(figsize=(5, 5))
patches, texts = plt.pie(
    x=types["SiteId"],
    colors=sns.color_palette("Set2")
)
plt.title(
    f"Observaciones de mosquitos de GLOBE en {county_name}: "
    "tipos generales de fuentes de agua"
)
plt.legend(
    patches,
    types["WaterSourceType"],
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False
)
plt.show()

In [ ]:
# Contar cada tipo específico de fuente de agua
types = (
    data_county[["SiteId", "WaterSource"]]
    .groupby("WaterSource", as_index=False)
    .count()
)

# Crear el gráfico circular
plt.figure(figsize=(5, 5))
patches, texts = plt.pie(
    x=types["SiteId"],
    colors=sns.color_palette("Set2")
)
plt.title(
    f"Observaciones de mosquitos de GLOBE en {county_name}: "
    "tipos específicos de fuentes de agua"
)
plt.legend(
    patches,
    types["WaterSource"],
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False
)
plt.show()